# Probabilistic distinguishing protocol — sweep-ready refactor

Each run of the protocol is now a single **function call**. Every input is an argument and every run builds its own seeded random generator, so no state leaks between runs (this removes the earlier hazard where `n` was silently overwritten by a later cell). This makes parameter sweeps safe.

Structure:
1. `calibrate_threshold(...)` — threshold from the "agree" (U) case
2. `run_protocol(...)` — runs calibration + trials, returns a summary dict
3. `append_run(...)` — appends one summary row to a CSV log
4. A single-run example, then a parameter sweep + plot.

In [ ]:
import numpy as np
import pandas as pd
import csv, os, datetime as dt
import matplotlib.pyplot as plt

In [ ]:
def calibrate_threshold(U, n, q, mu1, S_start, S_end, num_calib, percentile, rng):
    """Threshold from the 'agree' case: Alice draws from the U-distribution (mean mu1).

    Returns the `percentile`-th percentile of Bob's decision statistic |E - S/n|
    over `num_calib` trials in which Alice used distribution U.
    """
    vals = np.empty(num_calib)
    for i in range(num_calib):
        S = rng.integers(S_start, S_end + 1)
        bob_sum = rng.choice(U, size=q).sum()                 # q draws from public set U (with replacement)
        alice_sum = rng.exponential(scale=mu1, size=n - q).sum()
        A = (bob_sum + S + alice_sum) / n
        vals[i] = abs((A - mu1) - S / n)
    return float(np.percentile(vals, percentile))

In [ ]:
def run_protocol(n=700000, q=250, k=8, mu1=20000, mu2=50000,
                 S_start=10000, S_end=10_000_000,
                 num_trials=2000, num_calib=1500, percentile=80, seed=None):
    """Run one full experiment and return a summary dict (one notebook run = one call).

    A fresh np.random.Generator is created from `seed`, so runs are isolated and
    reproducible. Pass seed=None for a different draw each time.
    """
    rng = np.random.default_rng(seed)
    U = rng.exponential(scale=mu1, size=k)                     # public set, fixed for this run

    threshold = calibrate_threshold(U, n, q, mu1, S_start, S_end, num_calib, percentile, rng)

    successes = correct_U = total_U = correct_V = total_V = 0
    bob_tot = alice_tot = 0.0

    for _ in range(num_trials):
        bob_sum = rng.choice(U, size=q).sum()
        S = rng.integers(S_start, S_end + 1)
        bob_tx = bob_sum + S                                  # Bob -> Alice (S masks bob_sum)

        alice_choice = "U" if rng.random() < 0.5 else "V"
        m = mu1 if alice_choice == "U" else mu2
        alice_sum = rng.exponential(scale=m, size=n - q).sum()

        E = (alice_sum + bob_tx) / n - m
        guess = "U" if abs(E - S / n) <= threshold else "V"   # S cancels here by construction

        successes += (guess == alice_choice)
        if alice_choice == "U":
            total_U += 1; correct_U += (guess == "U")
        else:
            total_V += 1; correct_V += (guess == "V")
        bob_tot += bob_sum; alice_tot += alice_sum

    return {
        "success_rate":       successes / num_trials,
        "threshold":          threshold,
        "S_range":            f"{S_start}-{S_end}",
        "bob_to_alice_ratio": (bob_tot / num_trials) / (alice_tot / num_trials),
        "q":                  q,
        "n":                  n,
        "mu1":                mu1,
        "mu2":                mu2,
        "num_trials":         num_trials,
        # --- optional extras; delete these three if you only want the values above ---
        "acc_given_U":        correct_U / total_U if total_U else float("nan"),
        "acc_given_V":        correct_V / total_V if total_V else float("nan"),
        "seed":               seed,
    }

In [ ]:
def append_run(summary, log_path="notebook_run_log.csv"):
    """Append one summary dict as a row, writing the header only if the file is new."""
    record = {"timestamp": dt.datetime.now().isoformat(timespec="seconds"), **summary}
    write_header = not os.path.isfile(log_path)
    with open(log_path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(record.keys()))
        if write_header:
            w.writeheader()
        w.writerow(record)
    return record

## Single run
Run the protocol once and append the summary row to `notebook_run_log.csv`.

In [ ]:
summary = run_protocol(seed=0)     # defaults: n=700000, q=250, num_trials=2000
append_run(summary)
summary

## Parameter sweep

Loop over the settings you want to explore. Each combination is one isolated call and one logged row. Adjust the lists; keep per-run `num_trials`/`num_calib` modest while exploring, since a sweep runs the protocol many times.

In [ ]:
sweep_log = "sweep_log.csv"
if os.path.exists(sweep_log):
    os.remove(sweep_log)               # start fresh; comment out to keep accumulating across sweeps

for n in [200_000, 400_000, 700_000]:
    for q in [100, 250, 500]:
        summary = run_protocol(n=n, q=q, num_trials=1000, num_calib=800, seed=0)
        append_run(summary, log_path=sweep_log)
        print(f"n={n:>7}  q={q:>4}  success={summary['success_rate']:.3f}  "
              f"ratio={summary['bob_to_alice_ratio']:.2e}")

df = pd.read_csv(sweep_log)
df

## Plot: success rate vs. n, one line per q

In [ ]:
df = pd.read_csv(sweep_log)
fig, ax = plt.subplots(figsize=(7, 4))
for q_val, sub in df.groupby("q"):
    sub = sub.sort_values("n")
    ax.plot(sub["n"], sub["success_rate"], marker="o", label=f"q = {q_val}")
ax.axhline(0.5, ls="--", color="gray", lw=1, label="chance (0.5)")
ax.set_xlabel("n"); ax.set_ylabel("success rate")
ax.set_title("Success rate across the parameter sweep")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()